# Colab용 영화 리뷰 재분석기 (Ollama - A100/gemma4 고도화 버전)
이 노트북은 `tagged_reviews.json` 파일에서 룰베이스(Rule-based)로 임시 태깅된 리뷰들만 선별하여 다시 강력한 옵션과 함께 LLM을 통해 정교한 태깅을 재시도합니다.

In [3]:
# 1. 구글 드라이브 마운트
import os
import shutil
from google.colab import drive

mountpoint = '/content/drive'
if os.path.exists(mountpoint) and not os.path.ismount(mountpoint):
    shutil.rmtree(mountpoint)

drive.mount(mountpoint, force_remount=True)

Mounted at /content/drive


In [4]:
# 2. Ollama 설치 및 백그라운드 실행, 모델 다운로드
!apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

process = subprocess.Popen(['ollama', 'serve'])
time.sleep(3)

!pip install ollama

# 💡 고성능 처리를 위해 gemma4 모델 다운로드
!ollama pull gemma4

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (1,706 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current 

In [5]:
# 3. 라이브러리 임포트 및 경로 설정
import json
import os
import time
import re
import ollama

# 🏗️ 실제 본인의 구글 드라이브 경로에 맞게 수정하세요.
ROOT_DIR = "/content/drive/MyDrive/MovieReview/"
DATA_DIR = os.path.join(ROOT_DIR, "data")

LOCAL_MODEL = "gemma4"

IS_TEST_MODE = False
TEST_LIMIT = 20

print(f"데이터 경로 설정 완료: {DATA_DIR}")
print(f"사용 모델 설정 완료: {LOCAL_MODEL}")

데이터 경로 설정 완료: /content/drive/MyDrive/MovieReview/data
사용 모델 설정 완료: gemma4


In [6]:
# 4. 단일 처리용 LLM 함수 정의 (안정성 강화 버전)
def classify_single_review_with_local_llm(review):
    # ⚠️ 너무 긴 리뷰(에세이 등)는 모델 부하 및 서버 다운 방지를 위해 1500자로 제한합니다.
    content = review['content'][:1500]

    prompt = f"""
    당신은 영화 리뷰 분석 전문가입니다. 아래 리뷰를 읽고 범주에서 알맞은 태그를 선택하고, 핵심 키워드를 추출하세요.

    [범주]: 전체긍정, 전체부정, 전체복합, 연기좋음, 연기나쁨, 연출좋음, 연출나쁨, 서사좋음, 서사나쁨, 비주얼좋음, 비주얼나쁨, 음악좋음, 분위기가벼움, 분위기무거움, 고증좋음, 고증나쁨, 주의사항, TMI, 장르특성

    [지시사항]
    - 반드시 아래 JSON 객체 형식으로만 대답하세요. 다른 인사말이나 설명은 절대 넣지 마세요.
    - content_character 에는 범주에 해당하는 태그만 넣으세요.
    - search_keywords 에는 리뷰 본문에서 추출한 핵심 단어를 넣으세요.

    {{
        "content_character": ["태그1", "태그2"],
        "search_keywords": ["키워드1", "키워드2"]
    }}

    [리뷰 본문]:
    "{content}"
    """

    import time
    attempt = 0
    while True:
        try:
            attempt += 1
            # Ollama 서버와 통신
            response = ollama.chat(model=LOCAL_MODEL, messages=[
                {'role': 'user', 'content': prompt},
            ], format='json')

            if not response or 'message' not in response:
                raise ValueError("Ollama 서버로부터 빈 응답을 받았습니다.")

            response_text = response['message']['content']
            json_text = re.sub(r'```json|```', '', response_text).strip()

            if not json_text:
                raise ValueError("응답 텍스트가 비어있습니다.")

            res = json.loads(json_text)

            content_chars = res.get("content_character", [])
            keywords = res.get("search_keywords", [])

            if not content_chars or not keywords:
                raise ValueError("태그나 키워드가 비어있습니다.")

            allowed_tags = {"전체긍정", "전체부정", "전체복합", "연기좋음", "연기나쁨", "연출좋음", "연출나쁨", "서사좋음", "서사나쁨", "비주얼좋음", "비주얼나쁨", "음악좋음", "분위기가벼움", "분위기무거움", "고증좋음", "고증나쁨", "주의사항", "TMI", "장르특성"}
            res["content_character"] = [t for t in content_chars if t in allowed_tags]
            if not res["content_character"]:
                res["content_character"] = ["일반감상"]

            return res
        except Exception as e:
            # 서버 다운 또는 통신 오류 발생 시 잠시 대기 후 재시도
            print(f"⚠️ 분석 중 오류 발생 (시도 {attempt}): {e} -> 3초 후 재시도...")
            time.sleep(3)
            # 서버 프로세스가 죽었는지 체크는 못하지만, 보통 잠시 기다리면 리소스를 회수하고 다시 동작합니다.

In [7]:
# 5. 메인 실행 블록
from tqdm.notebook import tqdm
import os
import json

target_path = os.path.join(DATA_DIR, "tagged_reviews.json")

if not os.path.exists(target_path):
    print(f"❌ '{target_path}' 파일이 없습니다. 코랩 드라이브가 잘 마운트되었는지, 경로가 정확한지 확인해 주세요.")
else:
    with open(target_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    pending_reviews = []
    pending_refs = []

    # 룰베이스 및 아직 LLM 분석이 안 된 모든 항목 체크
    for m_name, m_data in data.items():
        for i, rev in enumerate(m_data.get("reviews", [])):
            method = rev.get("meta_tags", {}).get("analysis_method", "")
            if method == "Rule-based (Fallback)" or not method:
                pending_reviews.append(rev)
                pending_refs.append((m_name, i))

    if not pending_reviews:
        print("✅ 재분석할 항목이 없습니다! 모든 리뷰가 LLM으로 분석되었습니다.")
    else:
        if IS_TEST_MODE:
            pending_reviews = pending_reviews[:TEST_LIMIT]
            pending_refs = pending_refs[:TEST_LIMIT]
            print(f"⚠️ [테스트 모드 활성화] 리뷰 수를 {len(pending_reviews)}개로 제한합니다.")

        print(f"🚀 총 {len(pending_reviews)}개의 항목을 재분석합니다. (현재 드라이브 내 룰베이스 감지 개수: {len(pending_reviews)}개)")

        completed_count = 0
        start_time = time.time()

        for idx, rev in enumerate(tqdm(pending_reviews, desc="재분석 진행률")):
            res = classify_single_review_with_local_llm(rev)

            m_name, rev_idx = pending_refs[idx]
            data[m_name]["reviews"][rev_idx]["meta_tags"].update({
                "content_character": res.get("content_character", []),
                "search_keywords": res.get("search_keywords", []),
                "is_tmi": "TMI" in res.get("content_character", []),
                "has_warning": "주의사항" in res.get("content_character", []),
                "analysis_method": "Local-LLM"
            })

            completed_count += 1

            # 진행 정보 출력
            import datetime
            elapsed = time.time() - start_time
            eta_seconds = (elapsed / completed_count) * (len(pending_reviews) - completed_count)
            elapsed_str = str(datetime.timedelta(seconds=int(elapsed)))
            eta_str = str(datetime.timedelta(seconds=int(eta_seconds)))

            percentage = (completed_count / len(pending_reviews)) * 100
            if completed_count % 5 == 0:
                print(f"📦 진행 중... ({completed_count}/{len(pending_reviews)} - {percentage:.1f}%) | 소요: {elapsed_str} | 남은시간: {eta_str}")

            # 중간 저장
            if completed_count % 10 == 0:
                with open(target_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=4)

        # 최종 저장
        with open(target_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

        print(f"\n✨ 재분석 완료! (처리: {completed_count}개)")
        print(f"결과 저장: {target_path}")

🚀 총 6개의 항목을 재분석합니다. (현재 드라이브 내 룰베이스 감지 개수: 6개)


재분석 진행률:   0%|          | 0/6 [00:00<?, ?it/s]

📦 진행 중... (5/6 - 83.3%) | 소요: 0:03:24 | 남은시간: 0:00:40

✨ 재분석 완료! (처리: 6개)
결과 저장: /content/drive/MyDrive/MovieReview/data/tagged_reviews.json
